In [1]:
import cv2
import numpy as np
import os
import face_recognition as fr

In [2]:
path='Images'

In [3]:
os.listdir(path)

['Christie.jpeg',
 'Deepthi.jpeg',
 'Joseph.jpeg',
 'Kevin Manjila.jpg',
 'Kochuthreasia.jpeg',
 'Martin.jpeg',
 'Pushpam.jpeg']

In [4]:
my_list=os.listdir(path)
my_list

['Christie.jpeg',
 'Deepthi.jpeg',
 'Joseph.jpeg',
 'Kevin Manjila.jpg',
 'Kochuthreasia.jpeg',
 'Martin.jpeg',
 'Pushpam.jpeg']

In [5]:
imgs=[]
classnames=[]

for i in my_list:
    imgpath=os.path.join(path,i)
    img1=cv2.imread(imgpath)
    imgs.append(img1)
    classnames.append(i.split('.')[0])

In [6]:
def encodings(imgs):
    encoding_list=[]
    for i in imgs:
        cvt_image=cv2.cvtColor(i,cv2.COLOR_BGR2RGB)
        face_loc=fr.face_locations(cvt_image)
        face_encode=fr.face_encodings(cvt_image,face_loc)[0]
        encoding_list.append(face_encode)
        # print(face_encode)
    return encoding_list

In [7]:
encoding_list_known_face=encodings(imgs)

In [8]:
import csv
from datetime import datetime
import time

recorded_names=set()
show_status_time={}
status_delay=10
stop_camera=False

csv_file = "attendence.csv"

try:
    with open(csv_file,'r') as f:
        reader=csv.reader(f)
        for row in reader:
            if row:
                recorded_names.add(row[0])
    print("Loaded previous attendance:", recorded_names)
except FileNotFoundError:
    print("No previous attendance file found, creating new one.")




file=open("attendence.csv","a",newline="")
writer=csv.writer(file)

video=cv2.VideoCapture(0)

while True and not stop_camera:
    success,image=video.read()
    if not success:
        break
    cvt_image1=cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
    face_loc1=fr.face_locations(cvt_image1)
    face_encode1=fr.face_encodings(cvt_image1,face_loc1)
    for enc,loc in zip(face_encode1,face_loc1):
        y1,x2,y2,x1=loc
        matches=fr.compare_faces(encoding_list_known_face,enc)
        # print(matches)
        face_distance=fr.face_distance(encoding_list_known_face,enc)
        matchindex=np.argmin(face_distance)

        current_time=time.time()
        if matches[matchindex]==True:
            name=classnames[matchindex]
            if name not in show_status_time:
                show_status_time[name]=current_time
            if name in recorded_names:
                display_text1 = f'{name} Already marked'
                color = (0,255,255)
                cv2.rectangle(image,(x1,y1),(x2,y2),color,2)
                cv2.putText(image,display_text1,(x1,y1-10),cv2.FONT_HERSHEY_COMPLEX,0.5,color,2)
                
            if name not in recorded_names:
                # write to CSV
                timestamp=datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                writer.writerow([name,timestamp])
                file.flush()
                recorded_names.add(name)
                print("Attendance marked for:",name)
            
                duration=current_time-show_status_time[name]

                if duration<status_delay:
                    display_text=f'{name} Attendence Marked'
                    color=(0,255,0)
                    cv2.rectangle(image,(x1,y1),(x2,y2),color,2)
                    cv2.putText(image,display_text,(x1,y1-10),cv2.FONT_HERSHEY_COMPLEX,0.6,color,2)
                    cv2.imshow('Frame',image)
                    cv2.waitKey(5000)
                    stop_camera=True
                    break

        else:
            no_match="No matching images"
            cv2.rectangle(image,(x1,y1),(x2,y2),(0,0,255),3)
            cv2.putText(image,no_match,(x1,y1-10),cv2.FONT_HERSHEY_COMPLEX,1,(0,0,255),3)
            print(no_match)
    cv2.imshow('Frame',image)
    if cv2.waitKey(1)&0XFF==ord('q'):
        break

video.release()
file.close()
cv2.destroyAllWindows()



Loaded previous attendance: {'Kevin Manjila'}
